In [1]:
import MDAnalysis as mda
from MDAnalysis.analysis import contacts

import numpy as np
import pandas as pd

In [ ]:
# Fonction qui trouve le nombre d'atomes dans le peptide associé au WD40
def find_atoms_numbers(gro_file):
    
    with open(gro_file, "r") as f:
        lines = f.readlines()
    
    atom_lines = lines[2:-1]
    atoms = 0
    
    for line in atom_lines:
        atoms += 1
        if line.startswith("  166ARG      N"): #NOTE: ne fonctionne que pour le domaine WD40 murin de TBL1R (166-514)!
            return(int(atoms-1))


In [ ]:
def sel_atomes(u, protein_begin):
        
    protein_end = protein_begin + 5251 #NOTE: ne fonctionne que pour le domaine WD40 murin de TBL1R (166-514)!

    # Définis la partie MRD et WD40 
    MRD = u.atoms[0:protein_begin]
    WD40 = u.atoms[protein_begin:protein_end]

    # Sélection des résidus du MRD:
    sel_R328 = "(resid 328 and name NE NH* HE HH1* HH2*)"
    sel_K326 = "(resid 326 and name NZ HZ*)"
    sel_K330 = "(resid 330 and name NZ HZ*)"
    
    R328 = MRD.select_atoms(sel_R328)
    K326 = MRD.select_atoms(sel_K326)
    K330 = MRD.select_atoms(sel_K330) 
    
    # Sélection des résidus du WD40:
    sel_E171 = "(resid 171 and name OE*)"
    sel_D227 = "(resid 227 and name OD*)"
    partners_R328 = WD40.select_atoms(sel_E171 + " or " + sel_D227)
    
    sel_D313 = "(resid 313 and name OD*)"
    sel_N353_pep = "(resid 353 and name O)"
    sel_L312 = "(resid 312 and name O)"
    partners_K326 = WD40.select_atoms(sel_D313 + " or " + sel_N353_pep + " or " + sel_L312)
    
    sel_E351 = "(resid 351 and name OE*)"
    sel_N353 = "(resid 353 and name OD1)"
    sel_D369 = "(resid 369 and name OD*)"
    partners_K330 = WD40.select_atoms(sel_E351 + " or " + sel_N353 + " or " + sel_D369)
 
    
    return partners_R328, partners_K326, partners_K330, R328, K326, K330
    

In [4]:
# Fonction qui compte, pour un groupe de partenaires, combien de résidus distincts (pas d'atomes) sont en contact
def count_partner_residues_in_contact(contact_matrix_bool, partner_group):
    resids = partner_group.resids
    count = 0
    for resid in np.unique(resids):
        if contact_matrix_bool[resids == resid, :].any():
            count += 1
    return count

In [5]:
# Fonction qui calcule, en un seul passage de la trajectoire, le comptage par paires, l'occupation binaire
# ET le nombre de résidus partenaires distincts en contact (0-2 pour R328, 0-3 pour K326/K330), pour les 3 résidus
def search_contacts(partners_R328, partners_K326, partners_K330, R328, K326, K330, u_gro, radius=5.5):

    frames, times = [], []
    r328_pairs, k326_pairs, k330_pairs = [], [], []
    r328_binary, k326_binary, k330_binary = [], [], []
    r328_n_partners, k326_n_partners, k330_n_partners = [], [], []

    for ts in u_gro.trajectory:
        dist_R328 = contacts.distance_array(partners_R328.positions, R328.positions)
        dist_K326 = contacts.distance_array(partners_K326.positions, K326.positions)
        dist_K330 = contacts.distance_array(partners_K330.positions, K330.positions)

        cm_R328 = contacts.contact_matrix(dist_R328, radius)
        cm_K326 = contacts.contact_matrix(dist_K326, radius)
        cm_K330 = contacts.contact_matrix(dist_K330, radius)

        frames.append(ts.frame)
        times.append(ts.frame/100)

        r328_pairs.append(cm_R328.sum())
        k326_pairs.append(cm_K326.sum())
        k330_pairs.append(cm_K330.sum())

        r328_binary.append(int(cm_R328.any()))
        k326_binary.append(int(cm_K326.any()))
        k330_binary.append(int(cm_K330.any()))

        r328_n_partners.append(count_partner_residues_in_contact(cm_R328, partners_R328))
        k326_n_partners.append(count_partner_residues_in_contact(cm_K326, partners_K326))
        k330_n_partners.append(count_partner_residues_in_contact(cm_K330, partners_K330))

    merged_df = pd.DataFrame({
        "Frame": frames,
        "Time (ns)": times,
        "R328_Contacts_Pairs": r328_pairs,
        "R328_Contacts_Binary": r328_binary,
        "R328_Partners_Count": r328_n_partners,
        "K326_Contacts_Pairs": k326_pairs,
        "K326_Contacts_Binary": k326_binary,
        "K326_Partners_Count": k326_n_partners,
        "K330_Contacts_Pairs": k330_pairs,
        "K330_Contacts_Binary": k330_binary,
        "K330_Partners_Count": k330_n_partners,
    })

    return merged_df
    

In [6]:
# Fonction qui sélectionne les résidus et compte chaques intéraction sur toutes les différentes versions des trajectoires WT
def trajectory(xtc_path, gro_path, versions, mutation = False, name = [""]):
    if mutation == False:
        for version in versions:
            # Lance la trajectoire de MD
            GRO = gro_path+version+"/md_0_1.gro"
            XTC = xtc_path+version+"/md_0_1_noPBC.xtc"
            u_gro = mda.Universe(GRO, XTC)
    
            # Définition des atomes du MRD
            peptide_count = find_atoms_numbers(GRO)
            print(peptide_count)
    
            # Sélection des atomes 
            partners_R328, partners_K326, partners_K330, R328, K326, K330 = sel_atomes(u_gro, peptide_count)
    
            # Recherche des contacts et sauvegarde
            merged = search_contacts(partners_R328, partners_K326, partners_K330, R328, K326, K330, u_gro)
            merged.to_csv(f"{gro_path}{version}/Final_merged_contacts_{version}.csv", sep=";")
            print(f"DF sauvegardé dans: {gro_path}{version}/Final_merged_contacts_{version}.csv")

    else:
        for version in versions:
            for mut in name:
                # Lance la trajectoire de MD
                GRO = gro_path+version+"/"+mut+"/md_0_1.gro"
                XTC = xtc_path+version+"/"+mut+"/md_0_1_noPBC.xtc"
                u_gro = mda.Universe(GRO, XTC)

                # Définitions des atomes du MRD
                peptide_count = find_atoms_numbers(GRO)
                print(peptide_count)

                # Sélection des atomes
                partners_R328, partners_K326, partners_K330, R328, K326, K330 = sel_atomes(u_gro, peptide_count)

                # Recherche des contacts et sauvegarde
                merged = search_contacts(partners_R328, partners_K326, partners_K330, R328, K326, K330, u_gro)
                merged.to_csv(f"{gro_path}{version}/{mut}/Final_merged_contacts_{mut}_{version}.csv", sep=";")
                print(f"DF sauvegardé dans: {gro_path}{version}/{mut}/Final_merged_contacts_{mut}_{version}.csv")


In [7]:
# Fonction qui moyenne les DF merged
def average_df(path, mutation=False, name=[""]):

    if mutation == False:
        merged_df1 = pd.read_csv(f"{path}V1/Final_merged_contacts_V1.csv", sep=";", index_col=0)
        merged_df2 = pd.read_csv(f"{path}V2/Final_merged_contacts_V2.csv", sep=";", index_col=0)
        merged_df3 = pd.read_csv(f"{path}V3/Final_merged_contacts_V3.csv", sep=";", index_col=0)

        avg_df = pd.concat([merged_df1, merged_df2, merged_df3]).groupby(["Frame", "Time (ns)"], as_index=True).mean()
        avg_df.to_csv(f"{path}Final_averaged_contacts.csv", sep=";")
        print(f"DF moyenné dans: {path}Final_averaged_contacts.csv")        
    
    else:
        for mut in name:
            merged_df1 = pd.read_csv(f"{path}V1/{mut}/Final_merged_contacts_{mut}_V1.csv", sep=";", index_col=0).reset_index()
            merged_df2 = pd.read_csv(f"{path}V2/{mut}/Final_merged_contacts_{mut}_V2.csv", sep=";", index_col=0).reset_index()
            merged_df3 = pd.read_csv(f"{path}V3/{mut}/Final_merged_contacts_{mut}_V3.csv", sep=";", index_col=0).reset_index()
            
            avg_df = pd.concat([merged_df1, merged_df2, merged_df3]).groupby(["Frame", "Time (ns)"], as_index=True).mean()
            avg_df.to_csv(f"{path}Final_averaged_contacts_{mut}.csv", sep=";")
            print(f"DF moyenné dans: {path}Final_averaged_contacts_{mut}.csv")


In [8]:
# Variables:

In [9]:
input_path_wt = "E:/Trajectoire_MD/mNFIL3_100ns/WT/"
output_path_wt = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/"

input_path_mut = "E:/Trajectoire_MD/mNFIL3_100ns/Mutations/"
output_path_mut = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/"

input_path_wt_dist = "E:/Trajectoire_MD/mNFIL3_100ns/Distance/WT/"
output_path_wt_dist = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/"

input_path_mut_dist = "E:/Trajectoire_MD/mNFIL3_100ns/Distance/Mutations/"
output_path_mut_dist = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/"

In [10]:
replicas = ["V1", "V2", "V3"]
name_mutations = ["H325A", "I329A", "K326A", "K330A", "K332A", "L323A", "L327A", "P324A", "R328A", "S321A"]

In [11]:
# Lancement des fonctions

In [12]:
trajectory(input_path_wt, output_path_wt, replicas)
trajectory(input_path_mut, output_path_mut, replicas, True, name_mutations)

267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/V1/Final_merged_contacts_V1.csv
267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/V2/Final_merged_contacts_V2.csv
267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/V3/Final_merged_contacts_V3.csv
260
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/V1/H325A/Final_merged_contacts_H325A_V1.csv
258
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/V1/I329A/Final_merged_contacts_I329A_V1.csv
255
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/V1/K326A/

In [13]:
trajectory(input_path_wt_dist, output_path_wt_dist, replicas)
trajectory(input_path_mut_dist, output_path_mut_dist, replicas, mutation=True, name=name_mutations)

267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/V1/Final_merged_contacts_V1.csv
267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/V2/Final_merged_contacts_V2.csv
267
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/V3/Final_merged_contacts_V3.csv
260
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/V1/H325A/Final_merged_contacts_H325A_V1.csv
258
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/V1/I329A/Final_merged_contacts_I329A_V1.csv
255
DF sauvegardé dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL

In [14]:
average_df(output_path_wt)
average_df(output_path_mut, True, name_mutations)

DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/Final_averaged_contacts.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/Final_averaged_contacts_H325A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/Final_averaged_contacts_I329A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/Final_averaged_contacts_K326A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/Final_averaged_contacts_K330A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/Final_averaged_contacts_K332A.csv
DF moyenné dans: C:/Use

In [15]:
average_df(output_path_wt_dist)
average_df(output_path_mut_dist, mutation=True, name=name_mutations)

DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/Final_averaged_contacts.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/Final_averaged_contacts_H325A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/Final_averaged_contacts_I329A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/Final_averaged_contacts_K326A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/Final_averaged_contacts_K330A.csv
DF moyenné dans: C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/Fin